# 01 · Tensors you can reason about

**Goal:** translate familiar vector and matrix ideas into code whose dimensions you can explain.
Start with the setup and the first exercise. This is a skill check, not a speed test.

The exercises here were written for this course. ARENA's broader tensor exercises are available under
**Readings and official exercises** below. Use that source or PyTorch documentation when you need an API.

Run the next cell first. Expand **My progress** to see the tasks; tick them yourself after doing the work.

In [1]:
from pathlib import Path
import sys
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "curriculum.json").is_file())
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from workbench import lesson_panel, check
import torch
import einops
torch.set_num_threads(2)

display(lesson_panel("01"))

## Before running: predict the dimensions

A tensor's axes have meanings that you choose. Here, rows are examples and columns are features.
Predict the dimensions and values of each printed result. Then run the cell and explain any surprise.

In [2]:
examples = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
feature_offsets = torch.tensor([10., 20., 30.])
print("Examples:", examples.shape)
print(examples)
print("One vector of feature offsets:", feature_offsets.shape)
print("After adding the offsets:", (examples + feature_offsets).shape)
print(examples + feature_offsets)
print("Mean over examples:", examples.mean(dim=0).shape)
print("Mean over features:", examples.mean(dim=1).shape)

Examples: torch.Size([2, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.]])
One vector of feature offsets: torch.Size([3])
After adding the offsets: torch.Size([2, 3])
tensor([[11., 22., 33.],
        [14., 25., 36.]])
Mean over examples: torch.Size([3])
Mean over features: torch.Size([2])


**Your prediction and explanation:**

*Double-click this cell and write a few sentences. What did each axis represent?*

I'm going to assume that rows are individual examples and by features you mean the dimensions of each example, so this might be something like a house for each row and descriptors of said house for each column. 

From what I recall when you add two tensors, if the shapes don't match, broadcasting in torch changes the shapes of the tensors to make the operation possible. 

So, in this case, we'd duplicate the offsets vector and add it onto each row of examples, and we'd get things like 11, 22, 33, 14, 25, 36. The dimensions would stay the same. 

## 1. Put a sequence into batches

Implement `make_batches(values, batch_size)`.

- Input: a one-dimensional tensor of length `N` and a positive batch size that divides `N`.
- Output: shape `(N / batch_size, batch_size)`, preserving the original order.
- Preserve input values and dtype. Return a tensor without changing the input.
- For now, assume the stated inputs are valid.

Write the output dimensions on paper before coding. Use documentation freely.

<details><summary>Optional hint</summary>
Which operation changes the arrangement of dimensions while preserving the order of elements?
The checker includes a sliced input, so think about whether the input must be contiguous in memory.
</details>

In [9]:
def make_batches(values, batch_size):
    # Your implementation here.
    size = values.shape[0]
    return values.view(size // batch_size, batch_size)

In [10]:
batches_ok = check("batches", make_batches)

Case 1
  input (12,):
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
  batch size: 4
  expected (3, 4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
  received (3, 4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Case 2
  input (8,):
tensor([ 0,  2,  4,  6,  8, 10, 12, 14])
  batch size: 2
  expected (4, 2):
tensor([[ 0,  2],
        [ 4,  6],
        [ 8, 10],
        [12, 14]])
  received (4, 2):
tensor([[ 0,  2],
        [ 4,  6],
        [ 8, 10],
        [12, 14]])
PASS — now explain why it works before checking off the task.


Explanation: view is a pytorch command that lets you reshape tensors and matrices in place. Think of it as just changing the way torch interprets the dat (stride etc) which is stored contiguously in memory. 

## 2. Center each example independently

Implement `center_rows(x)` for a floating-point tensor of shape `(batch, features)`.
Subtract each example's own mean from its features. Return the same shape and preserve the input.

Predict what should happen to a row in which every feature has the same value.
The reading to consult is **Broadcasting** in ARENA 0.0.

<details><summary>Optional hint</summary>
Which axis contains the values that belong in one mean? What shape would let you combine each mean
with its own row, without accidentally combining different examples?
</details>

In [ ]:
def center_rows(x):
    # Your implementation here.
    return None

In [ ]:
center_ok = check("center", center_rows)

## 3. Compare directions

Implement `pairwise_cosine(x)` for rows representing vectors, with input shape `(examples, features)`.
Return a matrix where entry `(i, j)` is the cosine similarity of rows `i` and `j`.
For this exercise, define any pair involving a zero vector to have similarity zero.
Preserve the input. Predict the output shape, diagonal, and symmetry before coding.

<details><summary>Optional hint</summary>
How do vector length and direction contribute to a dot product? Which pairings do you need to compute?
How will your definition handle a zero-length vector?
</details>

In [37]:
def pairwise_cosine(x):
    # Your implementation here.
    #when dot producting transpose the remaining vectors I think

    #so if the input shape is examples x features
    #then the output shape would be examples x examples I expect
    #for each vector we'd dot product that vector with the entire input matrix, so a dot product of (1 x features) and (features x examples) makes a (1 x examples) output
    #and then if we do this for all examples rows then our output becomes (examples x examples)
    #the diagonals which would be the vectors and themselves are 1s 
    #I think that this should also be symmetric across the diagonal

    x_transpose = x.transpose(0, 1) #features x examples
    numerator = x @ x_transpose # examples x examples

    a = torch.linalg.vector_norm(x, dim=1, keepdim = True) #examples x 1
    b = torch.linalg.vector_norm(x_transpose, dim=0, keepdim = True) # 1 x examples

    denominator = (a @ b).clamp_min(1e-8)
    
    return numerator / denominator

In [38]:
similarity_ok = check("similarity", pairwise_cosine)

Case 1
  input (4, 2):
tensor([[1., 0.],
        [0., 2.],
        [1., 1.],
        [0., 0.]])
  expected (4, 4):
tensor([[1.0000, 0.0000, 0.7071, 0.0000],
        [0.0000, 1.0000, 0.7071, 0.0000],
        [0.7071, 0.7071, 1.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]])
  received (4, 4):
tensor([[1.0000, 0.0000, 0.7071, 0.0000],
        [0.0000, 1.0000, 0.7071, 0.0000],
        [0.7071, 0.7071, 1.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]])
Case 2
  input (2, 3):
tensor([[ 1., -1.,  0.],
        [-1.,  1.,  0.]])
  expected (2, 2):
tensor([[ 1., -1.],
        [-1.,  1.]])
  received (2, 2):
tensor([[ 1.0000, -1.0000],
        [-1.0000,  1.0000]])
PASS — now explain why it works before checking off the task.


## 4. Turn scores into stable probabilities

Implement `probabilities(logits)`. The last axis contains class scores; all earlier axes identify independent examples.
Return softmax probabilities with the same shape, using basic tensor operations rather than a built-in softmax.
Assume finite input scores. Your result should support automatic differentiation and work for large scores.

Examples of valid shapes are `(batch, classes)` and `(batch, positions, classes)`.
Explain why normalization must happen separately for each last-axis vector.

<details><summary>Optional hint</summary>
What happens to an exponential when its input is very large? Does adding the same constant to all scores
change their relative softmax probabilities? Which operations preserve the gradient connection?
</details>

In [ ]:
def probabilities(logits):
    # Your implementation here.
    return None

In [ ]:
probabilities_ok = check("probabilities", probabilities)

## Explain and vary

Pick one exercise. Predict what happens when you change the number of examples or features, then test it.
Write one bug you encountered and the conceptual misunderstanding behind it.

**My explanation:**

*Write here.*

**A variation I predicted and checked:**

*Write here.*

## Ready to move on?

You can explain each axis, solve the core tasks with documentation, and understand why the checks pass.
Mark completed tasks in the panel at the top, leave a stopping-point note, and save the notebook.

Continue to [02 · Train, measure, investigate](02_Train_a_Small_Model.ipynb), or return [home](../00_Start_Here.ipynb).